# Paper Replication: [TÍTULO DEL PAPER]

**Referencia completa:**  
Apellido, N. (AÑO). *Título completo del paper*. Journal, Vol(N), pp–pp.

**Pregunta central:**  
> _¿Qué pregunta responde el paper en una oración?_

**Intuición del modelo:**  
_Explica el mecanismo económico con tus propias palabras. Sin fórmulas todavía._

**Resultado principal:**  
_¿Qué encuentran? ¿Qué magnitud? ¿Es estadísticamente significativo?_

**Relevancia para Chile / tu contexto:**  
_¿Por qué importa esto para el mercado chileno, tu tesis, o Vicctus?_

In [ ]:
# ── 0. IMPORTS Y CONFIGURACIÓN ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_columns', 20)

# Semilla para reproducibilidad
np.random.seed(42)

print('Setup completo')

---
## 1. Datos

**Fuente original del paper:**  
- _¿Qué base de datos usaron? CRSP, Compustat, Bloomberg, CMF, BCCh..._

**Tu fuente equivalente:**  
- _¿Qué estás usando tú? ¿Hay diferencias relevantes en cobertura o frecuencia?_

**Muestra:**  
- Período: _YYYY–YYYY_  
- Universo: _N activos / mercado_  
- Frecuencia: _mensual / diaria_

In [ ]:
# ── 1. CARGA DE DATOS ──────────────────────────────────────────────────────
# Reemplaza con tu fuente real (CSV, API, Excel, etc.)
# df = pd.read_csv('data/retornos.csv', parse_dates=['fecha'], index_col='fecha')

# Ejemplo sintético (borra cuando tengas datos reales)
dates = pd.date_range('2010-01-31', '2024-12-31', freq='ME')
n = len(dates)
df = pd.DataFrame({
    'ret_mercado': np.random.normal(0.008, 0.045, n),
    'rf':          np.random.normal(0.003, 0.001, n),
    'smb':         np.random.normal(0.002, 0.030, n),
    'hml':         np.random.normal(0.003, 0.028, n),
}, index=dates)

print(f'Shape: {df.shape}')
print(f'Período: {df.index[0].date()} → {df.index[-1].date()}')
df.head()

In [ ]:
# ── 1b. ESTADÍSTICAS DESCRIPTIVAS  (replicar Tabla 1 del paper) ────────────
# Objetivo: tus números deben ser *comparables* a los del paper.
# Si divergen mucho, hay un problema de datos o construcción.

desc = df.describe().T
desc['sharpe_anual'] = (desc['mean'] * 12) / (desc['std'] * np.sqrt(12))
desc['media_anual_%'] = desc['mean'] * 12 * 100
desc['vol_anual_%']   = desc['std'] * np.sqrt(12) * 100

print('=== Estadísticas descriptivas (comparar con Tabla 1 del paper) ===')
desc[['media_anual_%', 'vol_anual_%', 'sharpe_anual', 'min', 'max']]

---
## 2. Construcción de variables

**Fórmulas exactas del paper:**  
Transcribe aquí las ecuaciones clave del paper usando LaTeX. Ejemplo para CAPM:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \varepsilon_{i,t}$$

Para Fama-French 3 factores:

$$R_{i,t} - R_{f,t} = \alpha_i + b_i \cdot MKT_t + s_i \cdot SMB_t + h_i \cdot HML_t + \varepsilon_{i,t}$$

**Decisiones de construcción:**  
- _¿Cómo defines el corte de size (mediana NYSE vs tu universo)?_  
- _¿Qué tasa libre de riesgo usas (BCU a 1 año, TPM, US T-bill)?_  
- _¿Cómo manejas los retornos extremos (winsorizing)?_

In [ ]:
# ── 2. CONSTRUCCIÓN DE VARIABLES ───────────────────────────────────────────
# Exceso de retorno sobre tasa libre de riesgo
df['mkt_excess'] = df['ret_mercado'] - df['rf']

# Aquí construirías tus factores específicos.
# Para Fama-French: sort por size y B/M, luego long-short portfolios.
# Para Nelson-Siegel: ajuste de curva con optimización.
# Para Brinson: weights × retornos de cada bucket sectorial.

# Ejemplo genérico: verificar correlaciones entre factores
factors = ['mkt_excess', 'smb', 'hml']
corr = df[factors].corr()

print('=== Correlación entre factores (deben ser bajas para ortogonalidad) ===')
print(corr.round(3))

# Visualización
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlación entre factores')
plt.tight_layout()
plt.show()

---
## 3. Modelo principal

**¿Qué estimación es la clave del paper?**  
_Describe el approach: time-series regression, cross-sectional regression, panel, GMM..._

**¿Qué tabla del paper estás tratando de replicar primero?**  
_Sé específico: "Tabla 2, Panel A — betas de los 25 portafolios"_

In [ ]:
# ── 3a. REGRESIÓN DE TIME SERIES ───────────────────────────────────────────
# Ejemplo: regresión de un portafolio contra los factores (like FF 1993 Table 2)

# Variable dependiente: exceso de retorno de algún activo/portafolio
# (aquí usamos mkt_excess como proxy; reemplaza con tu portafolio)
y = df['mkt_excess']  # <- tu portafolio aquí
X = sm.add_constant(df[['smb', 'hml']])

model = sm.OLS(y, X)
result = model.fit(cov_type='HAC', cov_kwds={'maxlags': 6})  # Newey-West

print(result.summary2())

In [ ]:
# ── 3b. TABLA DE RESULTADOS (formato publicable) ───────────────────────────
# Construir una tabla limpia como las del paper

def format_coef_table(result, name='Portafolio'):
    """Formatea coeficientes con t-stats entre paréntesis, como un paper."""
    rows = []
    for var in result.params.index:
        coef = result.params[var]
        tstat = result.tvalues[var]
        pval = result.pvalues[var]
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        rows.append({
            'Variable': var,
            'Coef.': f'{coef:.4f}{stars}',
            't-stat': f'({tstat:.2f})'
        })
    tbl = pd.DataFrame(rows).set_index('Variable')
    tbl.loc['R²'] = [f'{result.rsquared:.4f}', '']
    tbl.loc['N']  = [f'{int(result.nobs)}', '']
    return tbl

print('=== Tabla de resultados (replicar Tabla X del paper) ===')
format_coef_table(result, 'Mi portafolio')

---
## 4. Robustness checks

Los papers serios siempre incluyen verificaciones de robustez. Las más comunes:

- **Subperíodos:** ¿Se mantiene el resultado antes y después de una crisis?
- **Muestra alternativa:** ¿Funciona con datos de otro país o mercado?
- **Especificación alternativa:** ¿Cambia si usas retornos en exceso vs brutos?
- **Outliers:** ¿Qué pasa si winsorisas al 1% o excluyes crisis específicas?

In [ ]:
# ── 4. SUBPERÍODOS ────────────────────────────────────────────────────────
# Divide la muestra y re-estima el modelo en cada subperíodo

split = '2017-12-31'  # ajusta al punto de corte relevante para tu paper
periods = {
    f'Pre-{split[:4]}':  df.loc[:split],
    f'Post-{split[:4]}': df.loc[split:]
}

print(f'{'Período':<20} {'Alpha':>10} {'t(Alpha)':>12} {'R²':>8} {'N':>6}')
print('-' * 60)

for periodo, sub in periods.items():
    y_sub = sub['mkt_excess']
    X_sub = sm.add_constant(sub[['smb', 'hml']])
    res_sub = sm.OLS(y_sub, X_sub).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    alpha = res_sub.params['const']
    t_alpha = res_sub.tvalues['const']
    r2 = res_sub.rsquared
    n = int(res_sub.nobs)
    print(f'{periodo:<20} {alpha:>10.4f} {t_alpha:>12.2f} {r2:>8.4f} {n:>6}')

---
## 5. Insights y extensiones

### ¿Qué aprendí del paper?
_Escribe 3–5 puntos clave en tus propias palabras._

1. ...
2. ...
3. ...

### ¿Qué limitaciones tiene?
_¿Qué no puede responder este modelo? ¿Qué supuestos son cuestionables?_

### ¿Cómo lo extenderías?
_¿Qué harías diferente con datos chilenos, mercados emergentes, o tu contexto en Vicctus?_

### Papers relacionados que debería leer después:
- [ ] ...
- [ ] ...

---
*Notebook creado como parte del proceso de lectura activa de papers académicos.*